# 04 — Prompt avanzado: few-shot, CoT y JSON

**Level 1 — LLM Engineering**

Tres técnicas de prompt engineering en acción:

1. **Few-shot**: ejemplos en el prompt enseñan el formato esperado
2. **Chain-of-Thought**: pedir razonamiento paso a paso
3. **Salida JSON**: contrato estructurado parseable

In [1]:
import json

import requests

OLLAMA_HOST = "http://localhost:11434"


def chat(messages: list[dict], model: str = "llama3.2", temperature: float = 0.2) -> str:
    """Envia una conversacion completa a Ollama y devuelve la respuesta."""
    url = f"{OLLAMA_HOST}/api/chat"
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]["content"]

## 1 — Few-shot: ejemplos en el prompt

In [2]:
few_shot = [
    {
        "role": "system",
        "content": (
            "Clasifica el sentimiento de un texto: POSITIVO, NEGATIVO o NEUTRO. "
            "Responde solo con la palabra."
        ),
    },
    {"role": "user", "content": "Me encanta este curso"},
    {"role": "assistant", "content": "POSITIVO"},
    {"role": "user", "content": "El servicio es lento y malo"},
    {"role": "assistant", "content": "NEGATIVO"},
    {"role": "user", "content": "El clima esta normal hoy"},
    {"role": "assistant", "content": "NEUTRO"},
    {"role": "user", "content": "La pelicula fue increible, la recomiendo"},
]
print("=== 1. FEW-SHOT: clasificacion de sentimiento ===")
print(chat(few_shot))

=== 1. FEW-SHOT: clasificacion de sentimiento ===


POSITIVO


## 2 — Cadena de pensamiento: razonar paso a paso

In [3]:
cot = [
    {
        "role": "system",
        "content": (
            "Resuelve problemas paso a paso. "
            "Explica el razonamiento y termina con 'Respuesta: <resultado>'."
        ),
    },
    {
        "role": "user",
        "content": (
            "Una camiseta cuesta 25 euros. Tiene un descuento del 20% "
            "y ademas hay que sumar 3 euros de envio. ¿Cuanto se paga en total?"
        ),
    },
]
print("=== 2. CADENA DE PENSAMIENTO ===")
print(chat(cot))

=== 2. CADENA DE PENSAMIENTO ===


Paso 1: Primero, debemos calcular el descuento sobre el precio de la camiseta. El descuento es del 20% y la camiseta cuesta 25 euros.

Descuento = 25 x 0,20 = 5 euros

Paso 2: Ahora, debemos restar el descuento del precio original para obtener el precio con descuento.

Precio con descuento = 25 - 5 = 20 euros

Paso 3: Finalmente, debemos sumar el costo del envío para obtener el precio total.

Precio total = 20 + 3 = 23 euros

Respuesta: 23


## 3 — Salida estructurada: JSON

In [4]:
json_prompt = [
    {
        "role": "system",
        "content": (
            "Extrae la informacion del texto del usuario y devuelve "
            "SOLO un JSON valido con este formato: "
            '{"nombre": string, "edad": number, "ciudad": string}'
        ),
    },
    {"role": "user", "content": "Me llamo Ana, tengo 29 anos y vivo en Madrid"},
]
print("=== 3. SALIDA JSON ===")
respuesta = chat(json_prompt)
print(respuesta)
try:
    inicio = respuesta.find("{")
    fin = respuesta.rfind("}") + 1
    datos = json.loads(respuesta[inicio:fin])
    print()
    print("JSON parseado correctamente:")
    print(f"  nombre : {datos['nombre']}")
    print(f"  edad   : {datos['edad']}")
    print(f"  ciudad : {datos['ciudad']}")
except (json.JSONDecodeError, ValueError):
    print("El modelo no devolvio JSON valido.")

=== 3. SALIDA JSON ===


```json
{"nombre": "Ana", "edad": 29, "ciudad": "Madrid"}
```

JSON parseado correctamente:
  nombre : Ana
  edad   : 29
  ciudad : Madrid


## Conclusión

- **Few-shot** enseña formato con ejemplos (0-shot daba texto suelto).
- **CoT** fuerza el razonamiento explícito antes del resultado.
- **JSON** estructura la salida como contrato de máquina.